In [111]:
import dimod

In [112]:
cqm = dimod.ConstrainedQuadraticModel()

In [113]:
import random
import numpy as np
import pandas as pd
import yfinance as yf

def prepare_portfolio_data(number_of_stocks, years, interval="1d", random_state=None):
    """
    Randomly selects stocks, downloads historical closing prices from Yahoo Finance,
    cleans the data, calculates returns, expected returns, and covariance matrix.

    Returns
    -------
    selected_stocks : list
    prices : pandas.DataFrame
    returns : pandas.DataFrame
    expected_returns : pandas.Series
    covariance_matrix : pandas.DataFrame
    """

    # --------------------------------------------------
    # STOCK UNIVERSE
    # --------------------------------------------------

    stock_universe = [
        "AAPL", "MSFT", "GOOGL", "AMZN", "NVDA",
        "META", "TSLA", "AVGO", "JPM", "V",
        "MA", "WMT", "COST", "NFLX", "AMD",
        "ADBE", "CRM", "ORCL", "INTC", "QCOM",
        "CSCO", "IBM", "TXN", "AMAT", "MU",
        "PEP", "KO", "MCD", "SBUX", "NKE",
        "DIS", "HD", "LOW", "TGT", "CVX",
        "XOM", "COP", "BA", "CAT", "GE",
        "HON", "UPS", "FDX", "GS", "BAC",
        "MS", "AXP", "UNH", "JNJ", "PFE"
    ]

    # --------------------------------------------------
    # INPUT VALIDATION
    # --------------------------------------------------

    if not isinstance(number_of_stocks, int) or number_of_stocks <= 0:
        raise ValueError("number_of_stocks must be a positive integer.")

    if number_of_stocks > len(stock_universe):
        raise ValueError(
            f"Maximum number of stocks allowed is {len(stock_universe)}."
        )

    if not isinstance(years, int) or years <= 0:
        raise ValueError("years must be a positive integer.")

    if interval not in ["1d", "1wk"]:
        raise ValueError("interval must be either '1d' or '1wk'.")

    # --------------------------------------------------
    # RANDOM STOCK SELECTION
    # --------------------------------------------------

    rng = random.Random(random_state)

    selected_stocks = rng.sample(
        stock_universe,
        number_of_stocks
    )

    # --------------------------------------------------
    # DATE RANGE
    # --------------------------------------------------

    end_date = pd.Timestamp.today().normalize()
    start_date = end_date - pd.DateOffset(years=years)

    # --------------------------------------------------
    # DOWNLOAD DATA
    # --------------------------------------------------

    data = yf.download(
        tickers=selected_stocks,
        start=start_date,
        end=end_date,
        interval=interval,
        auto_adjust=False,
        progress=False,
        group_by="column",
        threads=True
    )

    if data.empty:
        raise ValueError("Yahoo Finance returned no data.")

    # --------------------------------------------------
    # EXTRACT CLOSE PRICES
    # --------------------------------------------------

    if isinstance(data.columns, pd.MultiIndex):

        if "Close" in data.columns.get_level_values(0):
            prices = data["Close"].copy()

        elif "Close" in data.columns.get_level_values(1):
            prices = data.xs(
                "Close",
                axis=1,
                level=1
            ).copy()

        else:
            raise ValueError("Close prices not found.")

    else:

        if "Close" not in data.columns:
            raise ValueError("Close prices not found.")

        prices = data[["Close"]].copy()

        if number_of_stocks == 1:
            prices.columns = selected_stocks

    # --------------------------------------------------
    # CHECK TICKERS
    # --------------------------------------------------

    if isinstance(prices, pd.Series):
        prices = prices.to_frame()

    missing_stocks = [
        stock
        for stock in selected_stocks
        if stock not in prices.columns
    ]

    if missing_stocks:
        raise ValueError(
            f"Data missing for these stocks: {missing_stocks}"
        )

    # Preserve order
    prices = prices.reindex(columns=selected_stocks)

    # --------------------------------------------------
    # CLEAN PRICES
    # --------------------------------------------------

    prices = prices.apply(
        pd.to_numeric,
        errors="coerce"
    )

    prices = prices.dropna(how="any")

    if len(prices) < 2:
        raise ValueError("Not enough valid historical data.")

    # --------------------------------------------------
    # RETURNS
    # --------------------------------------------------

    returns = prices.pct_change().dropna()

    # --------------------------------------------------
    # EXPECTED RETURNS
    # --------------------------------------------------

    expected_returns = returns.mean()

    # --------------------------------------------------
    # COVARIANCE MATRIX
    # --------------------------------------------------

    covariance_matrix = returns.cov()

    # Preserve ordering
    expected_returns = expected_returns.reindex(selected_stocks)

    covariance_matrix = covariance_matrix.reindex(
        index=selected_stocks,
        columns=selected_stocks
    )

    # --------------------------------------------------
    # RETURN MODEL INPUT DATA
    # --------------------------------------------------

    return (
        selected_stocks,
        prices,
        returns,
        expected_returns,
        covariance_matrix
    )

In [114]:
# ======================================================
# RUN FUNCTION
# ======================================================

(
    selected_stocks,
    prices,
    returns,
    expected_returns,
    covariance_matrix
) = prepare_portfolio_data(
    number_of_stocks= 4,
    years= 4,
    interval="1d",
    random_state=42
)

# ======================================================
# OUTPUT
# ======================================================

print("\n==============================")
print("SELECTED STOCKS")
print("==============================")
print(selected_stocks)

print("\nNumber of stocks:")
print(len(selected_stocks))

print("\n==============================")
print("HISTORICAL PRICES")
print("==============================")
display(prices.head())

print("\n==============================")
print("HISTORICAL RETURNS")
print("==============================")
display(returns.head())

print("\n==============================")
print("EXPECTED RETURNS")
print("==============================")
display(expected_returns)

print("\n==============================")
print("COVARIANCE MATRIX")
print("==============================")
display(covariance_matrix)

print("\n==============================")
print("DIMENSIONS")
print("==============================")

print("Prices:", prices.shape)
print("Returns:", returns.shape)
print("Expected Returns:", expected_returns.shape)
print("Covariance Matrix:", covariance_matrix.shape)

print("\n==============================")
print("MODEL INPUT DATA READY")
print("==============================")


SELECTED STOCKS
['HON', 'AVGO', 'MSFT', 'UNH']

Number of stocks:
4

HISTORICAL PRICES


Ticker,HON,AVGO,MSFT,UNH
Date,,,,
2022-09-09,189.479996,52.240002,264.459991,524.340027
2022-09-12,189.816086,52.905998,266.649994,531.250000
2022-09-13,182.847366,50.365002,251.990005,513.960022
2022-09-14,177.895126,51.075001,252.220001,509.769989
2022-09-15,174.820969,50.014000,245.380005,522.909973



HISTORICAL RETURNS


Ticker,HON,AVGO,MSFT,UNH
Date,,,,
2022-09-12,0.001774,0.012749,0.008281,0.013178
2022-09-13,-0.036713,-0.048029,-0.054978,-0.032546
2022-09-14,-0.027084,0.014097,0.000913,-0.008152
2022-09-15,-0.017281,-0.020773,-0.027119,0.025776
2022-09-16,0.002771,0.004719,-0.002608,-0.003614



EXPECTED RETURNS


Ticker
HON     0.000196
AVGO    0.002376
MSFT    0.000774
UNH    -0.000031
dtype: float64


COVARIANCE MATRIX


Ticker,HON,AVGO,MSFT,UNH
Ticker,,,,
HON,0.000204,0.000104,0.000058,0.000038
AVGO,0.000104,0.000856,0.000199,-0.000015
MSFT,0.000058,0.000199,0.000301,0.000021
UNH,0.000038,-0.000015,0.000021,0.000456



DIMENSIONS
Prices: (1002, 4)
Returns: (1001, 4)
Expected Returns: (4,)
Covariance Matrix: (4, 4)

MODEL INPUT DATA READY


In [115]:
BUDGET = 1000

In [116]:
A = dimod.Binary("Stock_A")
B = dimod.Binary("Stock_B")
C = dimod.Binary("Stock_C")
D = dimod.Binary("Stock_D")
#E = dimod.Binary("Stock_E")

x = [A, B, C, D]

In [117]:
mu = expected_returns.values
Sigma = covariance_matrix.values
price = prices.iloc[-1].values

portfolio_return = (
    mu[0] * A +
    mu[1] * B +
    mu[2] * C +
    mu[3] * D
    #mu[4] * E
)

portfolio_risk = sum(
    Sigma[i, j] * x[i] * x[j]
    for i in range(4)
    for j in range(4)
)

budget_constraint = sum(price[i] * x[i] for i in range(4))

In [118]:
price

array([208.24000549, 368.55999756, 493.95001221, 400.83999634])

In [119]:
risk_aversion = 0.5

objective = -(
    risk_aversion * portfolio_return
    -
    (1 - risk_aversion) * portfolio_risk
)

cqm.set_objective(objective)

In [120]:
cqm.add_constraint(
    budget_constraint <= BUDGET,
    label="budget"
)

'budget'

In [121]:
print(cqm)

Constrained quadratic model: 4 variables, 1 constraints, 14 biases

Objective
  3.823326717598011e-06*Binary('Stock_A') - 0.0007599759528082045*Binary('Stock_B') - 0.00023609870715316278*Binary('Stock_C') + 0.0002436462850738087*Binary('Stock_D') + 0.00010395524166954786*Binary('Stock_A')*Binary('Stock_B') + 5.841005712411989e-05*Binary('Stock_A')*Binary('Stock_C') + 0.0001988708349852741*Binary('Stock_B')*Binary('Stock_C') + 3.7557187088017184e-05*Binary('Stock_A')*Binary('Stock_D') - 1.4540945249161799e-05*Binary('Stock_B')*Binary('Stock_D') + 2.1248321482407586e-05*Binary('Stock_C')*Binary('Stock_D')

Constraints
  budget: 208.24000549316406*Binary('Stock_A') + 368.55999755859375*Binary('Stock_B') + 493.95001220703125*Binary('Stock_C') + 400.8399963378906*Binary('Stock_D') <= 1000.0

Bounds



In [122]:
sampler = dimod.ExactCQMSolver()

sampleset = sampler.sample_cqm(cqm)

In [123]:
print(sampleset)

   Stock_A Stock_B Stock_C Stock_D    energy num_oc. is_sat. is_fea.
5        0       1       1       0 -0.000797       1 arra... np.T...
1        0       1       0       0  -0.00076       1 arra... np.T...
3        1       1       0       0 -0.000652       1 arra... np.T...
7        1       1       1       0 -0.000631       1 arra... np.F...
13       0       1       1       1 -0.000547       1 arra... np.F...
9        0       1       0       1 -0.000531       1 arra... np.T...
11       1       1       0       1 -0.000386       1 arra... np.T...
15       1       1       1       1 -0.000343       1 arra... np.F...
4        0       0       1       0 -0.000236       1 arra... np.T...
6        1       0       1       0 -0.000174       1 arra... np.T...
0        0       0       0       0       0.0       1 arra... np.T...
2        1       0       0       0  0.000004       1 arra... np.T...
12       0       0       1       1  0.000029       1 arra... np.T...
14       1       0       1       1

In [124]:
feasible = sampleset.filter(
    lambda row: row.is_feasible
)

best = feasible.first

In [125]:
print("\nBest solution:")
print(best.sample)

print("\nObjective value:")
print(best.energy)


Best solution:
{'Stock_A': np.int64(0), 'Stock_B': np.int64(1), 'Stock_C': np.int64(1), 'Stock_D': np.int64(0)}

Objective value:
-0.0007972038249760931
